# ENGR 1451/2451 — Weekly Assignment 12
**10 points**

This assignment focuses on three goals:
(i) distinguishing statistical significance from practical importance;
(ii) comparing three population means with one-way ANOVA; and
(iii) interpreting adjusted pairwise comparisons and checking independence.

*All examples and the supplied CSV are constructed classroom data, not actual experiments or measurements.*

**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE12-1. A small improvement and a small $P$-value (3 pts)

Engineers compare the time needed to assemble a component with a **standard fixture** and a **new fixture**. The table below describes two separate, hypothetical randomized experiments. Different workers are independently assigned to the two fixtures, and each worker completes one assembly. Assume the large-sample normal approximation is appropriate.

Before collecting data, the team specifies a **two-sided** test of equal population mean times. A time saving of **at least 2 seconds per assembly** would justify the new fixture's cost.

### A. Calculate and compare the results (2 pts)

The experiments have the same sample means and SDs but different sample sizes. Let the estimated time saving be

$$\widehat\Delta=\bar x_{\mathrm{standard}}-\bar x_{\mathrm{new}}.$$

Use Python to create one results table with, for each experiment:

1. The observed time saving and its estimated standard error,

$$\widehat{\mathrm{SE}}(\widehat\Delta)=\sqrt{\frac{s_{\mathrm{standard}}^2}{n_{\mathrm{standard}}}+\frac{s_{\mathrm{new}}^2}{n_{\mathrm{new}}}}.$$

2. The test statistic $z=\widehat\Delta/\widehat{\mathrm{SE}}$, the approximate **two-sided** $P$-value using `2 * stats.norm.sf(abs(z))`, and the decision at $\alpha=0.05$.
3. The approximate 95% interval $\widehat\Delta\pm2\widehat{\mathrm{SE}}$. Use the lecture's multiplier **2**.

Report time quantities in **seconds**. Keep unrounded values for calculations; display three decimals for estimates and six decimals for $P$-values so a small value does not appear as zero.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.precision", 4)
alpha = 0.05

experiments = pd.DataFrame({
    "experiment": ["Smaller", "Larger"],
    "n_standard": [100, 1600],
    "n_new": [100, 1600],
    "mean_standard": [31.8, 31.8],
    "mean_new": [31.3, 31.3],
    "sd_standard": [4.0, 4.0],
    "sd_new": [4.0, 4.0],
})
display(experiments)

# Calculate the saving, estimated SE, z, two-sided P, and interval.
# Display a labeled results table with both experiments.

# Your code here

### B. Interpret the evidence (1 pt)

Answer in about **4–5 sentences**:

1. Why does the significance decision change even though the observed time saving does not? Does the smaller experiment establish that the true saving is zero?
2. Compare the larger experiment's estimated saving **and interval** with the 2-second requirement. Does its small $P$-value alone justify buying the fixture?
3. A colleague proposes switching to a one-sided test after seeing that the new fixture is faster. Explain why this conflicts with the planned analysis and why a $P$-value is not the probability that $H_0$ is true.

**Answer:**

## LE12-2. Which heat-sink designs cool differently? (4 pts)

The supplied file `./data/heat_sink_designs.csv` contains **36 constructed measurements**. Engineers independently prepare 36 electronic assemblies and randomly assign 12 to each heat-sink design: A, B, or C. Each assembly is tested once at the same power and ambient temperature. Treat measurements from different assemblies as independent.

| Column | Meaning |
|---|---|
| `assembly_id` | Unique classroom identifier |
| `design` | Heat-sink design: A, B, or C |
| `temp_rise_c` | Steady-state temperature rise above ambient (°C) |

Lower temperature rise indicates better cooling. Let $\mu_A$, $\mu_B$, and $\mu_C$ denote the population mean temperature rises under these test conditions.

### A. Summarize and inspect the groups (1 pt)

1. Display each design's sample size, mean, and sample SD using `groupby()` and `.agg(n="size", mean="mean", sd="std")`.
2. Make a boxplot of temperature rise by design, with a labeled vertical axis and units. Calculate the ratio of the largest to smallest group SD.
3. In **2–3 sentences**, discuss whether the spreads appear similar and whether there are obvious outliers. Explain why these small samples cannot establish normality and why independence must be justified from the study design.

**Answer:**

In [ ]:
cooling = pd.read_csv("./data/heat_sink_designs.csv")
display(cooling.head())
print(f"Assemblies: {len(cooling)}")
print(f"Unique IDs: {cooling['assembly_id'].nunique()}")

# Build cooling_summary using groupby("design")["temp_rise_c"].agg(...).
# Display the summary and largest/smallest SD ratio.
# Make a boxplot; label temperature rise in degrees Celsius.

# Your code here

### B. Build the ANOVA from squared deviations (2 pts)

Let $k$ be the number of designs and $N$ the total number of assemblies. Using the **individual measurements** and unrounded group means:

1. Calculate the grand mean $\bar y$ and all three sums of squares:

$$\mathrm{SSG}=\sum_j n_j(\bar y_j-\bar y)^2,\qquad
\mathrm{SSE}=\sum_j\sum_i(y_{ij}-\bar y_j)^2,\qquad
\mathrm{SST}=\sum_j\sum_i(y_{ij}-\bar y)^2.$$

   Check that $\mathrm{SST}\approx\mathrm{SSG}+\mathrm{SSE}$. The two sides may differ slightly because of floating-point arithmetic.
2. Calculate $df_G=k-1$, $df_E=N-k$, $\mathrm{MSG}=\mathrm{SSG}/df_G$, $\mathrm{MSE}=\mathrm{SSE}/df_E$, and $F=\mathrm{MSG}/\mathrm{MSE}$.
3. Find the **right-tail** $P$-value using `stats.f.sf(F, df_G, df_E)`. Display an ANOVA table with **Groups**, **Error**, and **Total** rows and columns for `df`, `sum_sq`, `mean_sq`, `F`, and `p_value`. Leave entries that do not apply blank or `NaN`.
4. Check your $F$ and $P$ against `stats.f_oneway(group_A, group_B, group_C)`, where each argument contains one design's individual measurements.

Keep `MSE`, `df_E`, and `cooling_summary` for Problem 3. Report sums of squares, mean squares, and $F$ to three decimals and $P$ to six decimals.

In [ ]:
# Calculate the grand mean and the group means from cooling.
# Hint: groupby("design")["temp_rise_c"].transform("mean")
# returns the appropriate group mean for every row.

# Calculate SSG, SSE, SST, degrees of freedom, MSG, MSE, F, and P.
# Display the ANOVA table and check SST = SSG + SSE.
# Select the three groups and check with stats.f_oneway(...).

# Your code here

### C. Explain the ANOVA (1 pt)

Answer in about **3–4 sentences**:

1. State $H_0$ and $H_A$ in terms of the population means and give the conclusion at $\alpha=0.05$, assuming the ANOVA conditions hold. Does rejection establish that **every pair** of population means differs?
2. Explain the $P$-value as a probability under the null hypothesis and the assumed model.
3. What variance does MSE estimate? Explain why ANOVA compares **mean squares**, rather than treating SSG and SSE themselves as variance estimates.

**Answer:**

## LE12-3. Follow-up comparisons and the unit of observation (3 pts)

### A. Which pairs have evidence of different means? (2 pts)

Investigate **all three pairs**: A–B, A–C, and B–C. Use the pooled error estimate from Problem 2, as in the lecture:

$$\mathrm{SE}_{ij}=\sqrt{\mathrm{MSE}\left(\frac{1}{n_i}+\frac{1}{n_j}\right)},\qquad
t_{ij}=\frac{\bar y_i-\bar y_j}{\mathrm{SE}_{ij}},\qquad df=df_E.$$

1. Build a table with each pair, its mean difference (first minus second), SE, $t$, and **two-sided** $P$-value. Use `2 * stats.t.sf(abs(t), df_E)` and the **same MSE and error degrees of freedom** for every pair.
2. Compute the number of comparisons $K=k(k-1)/2$ and the Bonferroni cutoff $\alpha_{\mathrm{each}}=0.05/K$. Add columns showing significance at the ordinary 0.05 cutoff and at the Bonferroni cutoff. Compare unrounded $P$-values.
3. In **3–4 sentences**, identify which pairs survive the adjustment, which decision changes, and which design has the lower sample mean for each surviving comparison. Explain why a nonsignificant pairwise result does not establish equal population means.

**Answer:**

In [ ]:
pairs = [("A", "B"), ("A", "C"), ("B", "C")]

# Use cooling_summary, MSE, and df_E from Problem 2.
# For each pair, calculate mean difference, pooled SE, t, and two-sided P.
# Add the ordinary and Bonferroni significance decisions.
# Do not recompute a separate variance estimate for each pair.

# Your code here

### B. Would repeated readings give the same information? (1 pt)

Consider a **different** experiment: the team builds only **one assembly per design**, then records its temperature 12 times. The resulting file still has 36 rows. A colleague proposes analyzing it as though there were 12 independent assemblies per design.

In **3–4 sentences**, identify the experimental units and the actual number of independently tested assemblies per design. Explain why repeated readings from the same assembly do not provide independent replication of the design, why the ANOVA error estimate and $P$-value from that proposed analysis would be questionable, and how you would redesign the experiment.

**Answer:**

## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*

## Before you submit

1. Keep `LE12.ipynb` next to a `data` folder containing `LE12_heat_sink_designs.csv`. Extract the complete student ZIP to preserve this arrangement.
2. **Restart Kernel and Run All Cells** after completing the notebook and confirm every cell runs without errors.
3. Confirm your calculated tables, plot, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.